# ============================================================================
# EVERSYS LEGACY COFFEE MACHINE - DATA SCHEMA
# ============================================================================

"""
SENSOR DATA SCHEMA FOR LEGACY COFFEE MACHINE
Based on Eversys Legacy User Manual (Published 29.06.2022)

This schema tracks all operational parameters, sensors, and machine states
for predictive maintenance, quality control, and operational analytics.
"""

from dataclasses import dataclass
from typing import Optional, List
from datetime import datetime
from enum import Enum

# ============================================================================
# ENUMERATIONS
# ============================================================================

class MachineModel(Enum):
    L2C = "L'2c"      # Coffee, hot water, powder
    L2M = "L'2m"      # Coffee, hot water, milk, powder
    L2S = "L'2s"      # Coffee, hot water, steam, powder
    L2MS = "L'2ms"    # Coffee, hot water, steam, milk, powder

class MachineState(Enum):
    OFF = "off"
    INITIALIZING = "initializing"
    HEATING = "heating"
    READY = "ready"
    DISPENSING = "dispensing"
    RINSING = "rinsing"
    CLEANING = "cleaning"
    STANDBY = "standby"
    ERROR = "error"

class ProductType(Enum):
    RISTRETTO = "ristretto"
    ESPRESSO = "espresso"
    COFFEE = "coffee"
    AMERICANO = "americano"
    CAPPUCCINO = "cappuccino"
    LATTE = "latte"
    LATTE_MACCHIATO = "latte_macchiato"
    HOT_WATER = "hot_water"
    STEAM = "steam"
    CHOCO = "choco"

class ErrorSeverity(Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"

# ============================================================================
# SENSOR DATA STRUCTURES
# ============================================================================

@dataclass
class BoilerSensors:
    """Coffee and Steam Boiler Monitoring"""
    # Coffee Boiler (0.6L capacity)
    coffee_temperature_celsius: float  # Target: 90-96°C
    coffee_pressure_bar: float         # Target: 9 bar for extraction
    coffee_water_level_percent: float  # 0-100%
    coffee_heating_element_on: bool
    
    # Steam Boiler (0.8L capacity) - for models with steam
    steam_temperature_celsius: Optional[float] = None  # Target: ~130°C
    steam_pressure_bar: Optional[float] = None         # Target: 1.2-1.5 bar
    steam_water_level_percent: Optional[float] = None  # 0-100%
    steam_heating_element_on: Optional[bool] = None

@dataclass
class GrinderSensors:
    """Up to 4 Grinders (ceramic burrs 64mm)"""
    grinder_id: int                    # 1-4 (positions 1-4)
    motor_speed_rpm: float             # Rotations per minute
    motor_temperature_celsius: float   # Should stay cool
    grind_size_mm: float              # Adjustment in 0.01mm increments
    coffee_dose_grams: float          # Amount ground (target ~24g for double)
    is_active: bool
    bean_hopper_level_percent: float  # 0-100%, alerts at low level
    burr_wear_cycles: int             # Track for maintenance

@dataclass
class BrewingChamberSensors:
    """Single 24g brewing chamber"""
    chamber_position: str              # "up", "down", "brewing"
    top_piston_position_mm: float
    bottom_piston_position_mm: float
    chamber_pressure_bar: float        # During extraction
    chamber_temperature_celsius: float
    cake_thickness_mm: float           # Compacted coffee puck
    extraction_time_seconds: float     # Ideal: 21-23s for espresso
    water_flow_ml_per_sec: float      # Flow rate monitoring

@dataclass
class MilkSystemSensors:
    """Milk system (for models with fridge)"""
    fridge_temperature_celsius: float  # Target: <3°C (37°F)
    fridge_door_open: bool
    milk_tank_level_ml: float         # Capacity varies
    milk_temperature_celsius: float    # During frothing
    milk_pump_active: bool
    foam_texture_setting: int          # 1-10 scale
    milk_tube_flow_rate_ml_per_sec: float

@dataclass
class PowderSystemSensors:
    """Powder/Choco unit (optional)"""
    container_1_level_percent: float   # Powder container 1 (1kg capacity)
    container_2_level_percent: float   # Powder container 2 (1kg capacity)
    mixer_speed_rpm: float
    mixer_temperature_celsius: float
    dispense_amount_grams: float

@dataclass
class WaterSystemSensors:
    """Water supply and drainage"""
    inlet_pressure_bar: float          # Target: 2.5-4 bar
    inlet_flow_rate_lph: float        # Liters per hour, min 140 L/h
    inlet_temperature_celsius: float
    water_hardness_dgh: float         # Target: 5-8°dGH
    water_filter_life_percent: float   # 0-100%
    drain_flow_active: bool

@dataclass
class GroundsDrawerSensors:
    """Waste management"""
    grounds_weight_grams: float        # Capacity ~400g
    grounds_count: int                 # Number of pucks
    drawer_present: bool
    drawer_full: bool                  # Alert trigger

@dataclass
class TouchScreenSensors:
    """User interface monitoring"""
    screen_active: bool
    brightness_level_percent: int      # 0-100%
    current_menu: str
    button_presses_total: int
    last_interaction_timestamp: datetime

@dataclass
class ElectricalSensors:
    """Power and electrical monitoring"""
    voltage_volts: float               # 220-240V (region dependent)
    current_amps: float               # Max 16A
    power_consumption_watts: float     # Up to 2300W
    total_energy_kwh: float           # Cumulative energy use
    main_switch_on: bool
    standby_mode: bool

@dataclass
class EnvironmentalSensors:
    """Ambient conditions"""
    ambient_temperature_celsius: float # Operating: 10-32°C
    ambient_humidity_percent: float    # Operating: 5-80% RH
    machine_vibration_level: float     # Accelerometer data

@dataclass
class CleaningSensors:
    """Cleaning system monitoring"""
    cleaning_ball_dispenser_level: int      # Count remaining
    milk_cleaning_ball_dispenser_level: int # Count remaining
    last_cleaning_timestamp: datetime
    cleaning_cycles_since_service: int
    rinse_counter: int

# ============================================================================
# OPERATIONAL DATA
# ============================================================================

@dataclass
class ProductionMetrics:
    """Product counters and production data"""
    total_products_dispensed: int
    products_by_type: dict             # {ProductType: count}
    average_extraction_time_sec: float
    products_since_cleaning: int
    products_since_service: int
    daily_production_count: int
    peak_hour_production: int

@dataclass
class MaintenanceData:
    """Maintenance tracking"""
    total_runtime_hours: float
    last_service_date: datetime
    last_service_product_count: int
    water_filter_replacement_date: datetime
    burr_replacement_date: datetime
    next_service_due_products: int     # Alert at 50,000

@dataclass
class ErrorLog:
    """Error and warning tracking"""
    error_code: str                    # E-XXX, S-XXX, W-XXX
    error_severity: ErrorSeverity
    error_message: str
    timestamp: datetime
    resolved: bool
    resolution_time_minutes: Optional[int]

# ============================================================================
# COMPLETE MACHINE STATE
# ============================================================================

@dataclass
class CoffeeMachineState:
    """Complete machine state snapshot"""
    # Identification
    machine_id: str
    machine_model: MachineModel
    serial_number: str
    firmware_version: str
    timestamp: datetime
    
    # Operational State
    machine_state: MachineState
    uptime_hours: float
    
    # All Sensor Systems
    boilers: BoilerSensors
    grinders: List[GrinderSensors]     # Up to 4
    brewing_chamber: BrewingChamberSensors
    milk_system: Optional[MilkSystemSensors]
    powder_system: Optional[PowderSystemSensors]
    water_system: WaterSystemSensors
    grounds_drawer: GroundsDrawerSensors
    touchscreen: TouchScreenSensors
    electrical: ElectricalSensors
    environmental: EnvironmentalSensors
    cleaning: CleaningSensors
    
    # Operational Metrics
    production: ProductionMetrics
    maintenance: MaintenanceData
    active_errors: List[ErrorLog]
    
    # Current Operation
    current_product: Optional[ProductType]
    product_start_time: Optional[datetime]


# ============================================================================
# SYNTHETIC DATA GENERATION PSEUDOCODE
# ============================================================================

"""
PSEUDOCODE FOR SYNTHETIC DATA GENERATION

Purpose: Generate realistic time-series sensor data for the Legacy coffee machine
         to enable analytics, ML model training, and predictive maintenance.

Key Considerations:
- Realistic sensor ranges based on manual specifications
- Time-based patterns (morning rush, afternoon lull, cleaning cycles)
- Wear and degradation over time
- Correlated sensor readings (e.g., more products = fuller grounds drawer)
- Occasional faults and errors
- Seasonal variations
"""

# ============================================================================
# INITIALIZATION
# ============================================================================

FUNCTION initialize_synthetic_machine():
    """
    Set up a new synthetic machine with initial parameters
    """
    machine = CoffeeMachineState()
    
    # Random machine ID and configuration
    machine.machine_id = generate_uuid()
    machine.machine_model = random_choice([L2C, L2M, L2S, L2MS])
    machine.serial_number = generate_serial_number()
    machine.firmware_version = "V3.17"
    
    # Initialize all sensors to normal operating values
    machine.boilers = initialize_boilers_normal()
    machine.grinders = initialize_grinders(count=random_int(1, 4))
    machine.brewing_chamber = initialize_brewing_chamber()
    machine.water_system = initialize_water_system()
    machine.grounds_drawer = initialize_grounds_drawer()
    
    # Conditionally initialize based on model
    IF machine.machine_model IN [L2M, L2MS]:
        machine.milk_system = initialize_milk_system()
    
    IF has_powder_option:
        machine.powder_system = initialize_powder_system()
    
    # Set to morning startup state
    machine.machine_state = MachineState.INITIALIZING
    machine.timestamp = get_business_hours_start()
    
    RETURN machine


# ============================================================================
# MAIN SIMULATION LOOP
# ============================================================================

FUNCTION simulate_machine_operation(duration_days, sample_rate_seconds):
    """
    Main simulation loop - generates time-series data
    
    Args:
        duration_days: How many days to simulate
        sample_rate_seconds: Data recording frequency (e.g., 30 seconds)
    """
    
    machine = initialize_synthetic_machine()
    current_time = machine.timestamp
    end_time = current_time + duration_days * 24 * 3600
    
    data_points = []
    
    WHILE current_time < end_time:
        
        # Determine time of day context
        hour_of_day = get_hour(current_time)
        is_business_hours = is_business_time(current_time)
        is_morning_rush = (hour_of_day BETWEEN 7 AND 10)
        is_afternoon_peak = (hour_of_day BETWEEN 14 AND 16)
        is_cleaning_time = (hour_of_day == 22)  # End of day
        
        # State transitions based on time and conditions
        IF is_cleaning_time AND products_since_cleaning > 100:
            transition_to_cleaning(machine)
        
        ELSE IF NOT is_business_hours:
            IF machine.machine_state != MachineState.STANDBY:
                transition_to_standby(machine)
        
        ELSE IF is_business_hours AND machine.machine_state == MachineState.STANDBY:
            transition_to_startup(machine)
        
        ELSE IF machine.machine_state == MachineState.READY:
            # Probabilistic product dispensing
            dispense_probability = calculate_dispense_probability(
                hour_of_day, 
                is_morning_rush, 
                is_afternoon_peak
            )
            
            IF random() < dispense_probability:
                product_type = select_random_product_weighted()
                dispense_product(machine, product_type)
        
        # Update all sensors based on current state
        update_all_sensors(machine, current_time)
        
        # Simulate wear and degradation
        apply_degradation(machine)
        
        # Occasionally inject faults/errors
        IF random() < FAULT_PROBABILITY:
            inject_random_fault(machine)
        
        # Record snapshot
        snapshot = deep_copy(machine)
        snapshot.timestamp = current_time
        data_points.append(snapshot)
        
        # Advance time
        current_time = current_time + sample_rate_seconds
    
    RETURN data_points


# ============================================================================
# SENSOR UPDATE FUNCTIONS
# ============================================================================

FUNCTION update_all_sensors(machine, current_time):
    """
    Update all sensor readings based on machine state and operations
    """
    
    # BOILERS
    IF machine.machine_state == MachineState.HEATING:
        # Temperature rises toward target
        machine.boilers.coffee_temperature_celsius += random_normal(0.5, 0.1)
        machine.boilers.coffee_temperature_celsius = min(96, machine.boilers.coffee_temperature_celsius)
        machine.boilers.coffee_heating_element_on = TRUE
        
        IF machine.boilers.steam_temperature_celsius IS NOT NULL:
            machine.boilers.steam_temperature_celsius += random_normal(1.0, 0.2)
            machine.boilers.steam_heating_element_on = TRUE
    
    ELSE IF machine.machine_state == MachineState.READY:
        # Maintain temperature with small fluctuations
        machine.boilers.coffee_temperature_celsius = add_noise(93, 1.5)
        machine.boilers.coffee_heating_element_on = cycle_heating_element()
        
        IF machine.boilers.steam_temperature_celsius IS NOT NULL:
            machine.boilers.steam_temperature_celsius = add_noise(130, 3)
    
    ELSE IF machine.machine_state == MachineState.DISPENSING:
        # Temperature drops during brewing
        machine.boilers.coffee_temperature_celsius -= random_uniform(0.5, 2.0)
        machine.boilers.coffee_water_level_percent -= 5
    
    # GRINDERS
    FOR EACH grinder IN machine.grinders:
        IF grinder.is_active:
            grinder.motor_speed_rpm = random_normal(1500, 50)
            grinder.motor_temperature_celsius = random_normal(35, 5)
            grinder.bean_hopper_level_percent -= random_uniform(0.1, 0.3)
            grinder.burr_wear_cycles += 1
        ELSE:
            grinder.motor_speed_rpm = 0
            grinder.motor_temperature_celsius -= 0.1  # Cooling
    
    # BREWING CHAMBER
    IF machine.machine_state == MachineState.DISPENSING:
        simulate_extraction_cycle(machine.brewing_chamber)
    ELSE:
        machine.brewing_chamber.chamber_pressure_bar = 0
        machine.brewing_chamber.water_flow_ml_per_sec = 0
    
    # MILK SYSTEM
    IF machine.milk_system IS NOT NULL:
        machine.milk_system.fridge_temperature_celsius = add_noise(2.5, 0.5)
        IF dispensing_milk_product:
            machine.milk_system.milk_tank_level_ml -= product_milk_volume
            machine.milk_system.milk_pump_active = TRUE
        ELSE:
            machine.milk_system.milk_pump_active = FALSE
    
    # WATER SYSTEM
    machine.water_system.inlet_pressure_bar = add_noise(3.0, 0.3)
    machine.water_system.inlet_temperature_celsius = add_noise(15, 2)
    machine.water_system.water_filter_life_percent -= 0.0001  # Slow degradation
    
    # GROUNDS DRAWER
    IF product_just_dispensed:
        machine.grounds_drawer.grounds_count += 1
        machine.grounds_drawer.grounds_weight_grams += random_normal(18, 2)
        IF machine.grounds_drawer.grounds_count >= 20:
            machine.grounds_drawer.drawer_full = TRUE
    
    # ELECTRICAL
    base_power = 300  # Standby
    IF machine.machine_state == MachineState.HEATING:
        base_power = 2200
    ELSE IF machine.machine_state == MachineState.DISPENSING:
        base_power = 1800
    ELSE IF machine.machine_state == MachineState.READY:
        base_power = 600
    
    machine.electrical.power_consumption_watts = add_noise(base_power, 50)
    machine.electrical.total_energy_kwh += (base_power / 1000) * (sample_rate_seconds / 3600)
    
    # ENVIRONMENTAL
    machine.environmental.ambient_temperature_celsius = add_noise(22, 2)
    machine.environmental.ambient_humidity_percent = add_noise(50, 10)


FUNCTION simulate_extraction_cycle(brewing_chamber):
    """
    Detailed simulation of coffee extraction process
    """
    # Typical espresso extraction: 21-23 seconds
    elapsed_time = brewing_chamber.extraction_time_seconds
    
    IF elapsed_time < 3:
        # Pre-infusion phase
        brewing_chamber.chamber_pressure_bar = linear_ramp(0, 2, elapsed_time, 3)
        brewing_chamber.water_flow_ml_per_sec = 0.5
    
    ELSE IF elapsed_time < 23:
        # Main extraction phase
        brewing_chamber.chamber_pressure_bar = add_noise(9, 0.5)
        brewing_chamber.water_flow_ml_per_sec = random_normal(2.0, 0.2)
        brewing_chamber.chamber_temperature_celsius = add_noise(93, 1)
    
    ELSE:
        # Post-extraction
        brewing_chamber.chamber_pressure_bar = exponential_decay(9, elapsed_time - 23)
        brewing_chamber.water_flow_ml_per_sec = 0
    
    brewing_chamber.extraction_time_seconds += sample_rate_seconds


# ============================================================================
# PRODUCT DISPENSING
# ============================================================================

FUNCTION dispense_product(machine, product_type):
    """
    Simulate dispensing a specific product
    """
    machine.machine_state = MachineState.DISPENSING
    machine.current_product = product_type
    machine.product_start_time = current_time
    
    # Select appropriate grinder and activate
    grinder_id = select_grinder_for_product(product_type)
    machine.grinders[grinder_id].is_active = TRUE
    
    # Set brewing parameters based on product
    IF product_type == ProductType.ESPRESSO:
        brewing_time = random_normal(22, 1)
        coffee_dose = random_normal(9, 0.5)
        water_volume = random_normal(30, 2)
    
    ELSE IF product_type == ProductType.CAPPUCCINO:
        brewing_time = random_normal(23, 1)
        coffee_dose = random_normal(18, 0.5)
        water_volume = random_normal(40, 2)
        milk_volume = random_normal(150, 10)
        
        IF machine.milk_system IS NOT NULL:
            machine.milk_system.milk_tank_level_ml -= milk_volume
    
    # Update production metrics
    machine.production.total_products_dispensed += 1
    machine.production.products_by_type[product_type] += 1
    machine.production.products_since_cleaning += 1
    machine.production.products_since_service += 1
    
    # Transition back to ready after product complete
    SCHEDULE transition_to_ready(machine) AFTER brewing_time


# ============================================================================
# DEGRADATION AND WEAR
# ============================================================================

FUNCTION apply_degradation(machine):
    """
    Simulate gradual wear and component degradation
    """
    
    # Grinder burr wear
    FOR EACH grinder IN machine.grinders:
        IF grinder.burr_wear_cycles > 500000:  # Significant wear
            # Grind size becomes less consistent
            grinder.grind_size_mm += random_normal(0, 0.001)
    
    # Water filter degradation
    IF machine.water_system.water_filter_life_percent < 10:
        # Water hardness increases
        machine.water_system.water_hardness_dgh += random_normal(0, 0.1)
    
    # Heating element efficiency loss over time
    IF machine.maintenance.total_runtime_hours > 5000:
        # Takes longer to heat
        heating_efficiency_factor = 0.95


# ============================================================================
# FAULT INJECTION
# ============================================================================

FUNCTION inject_random_fault(machine):
    """
    Randomly inject realistic faults based on manual error codes
    """
    fault_type = weighted_random_choice([
        ("bean_hopper_empty", 0.15),
        ("grounds_drawer_full", 0.20),
        ("low_water_pressure", 0.10),
        ("milk_tank_empty", 0.12),
        ("grinder_blocked", 0.05),
        ("temperature_sensor_error", 0.03),
        ("cleaning_required", 0.25),
        ("minor_sensor_noise", 0.10)
    ])
    
    IF fault_type == "bean_hopper_empty":
        hopper_id = random_choice(machine.grinders)
        hopper_id.bean_hopper_level_percent = 0
        add_error(machine, "S-010", "Bean hopper empty", ErrorSeverity.LOW)
    
    ELSE IF fault_type == "grounds_drawer_full":
        machine.grounds_drawer.drawer_full = TRUE
        add_error(machine, "S-006", "Grounds drawer full", ErrorSeverity.LOW)
    
    ELSE IF fault_type == "low_water_pressure":
        machine.water_system.inlet_pressure_bar = 1.5  # Below min 2.5 bar
        add_error(machine, "W-004", "Water flow too low", ErrorSeverity.MEDIUM)
    
    # ... (implement other fault types)


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

FUNCTION calculate_dispense_probability(hour, is_rush, is_peak):
    """
    Calculate probability of product being dispensed at current time
    """
    base_probability = 0.05  # 5% per time sample during business hours
    
    IF is_rush:
        RETURN base_probability * 4  # 20% - very busy
    ELSE IF is_peak:
        RETURN base_probability * 2  # 10% - moderately busy
    ELSE IF hour BETWEEN 11 AND 13:  # Lunch
        RETURN base_probability * 2.5
    ELSE:
        RETURN base_probability

FUNCTION select_random_product_weighted():
    """
    Select product type based on realistic consumption patterns
    """
    RETURN weighted_random_choice([
        (ProductType.ESPRESSO, 0.25),
        (ProductType.CAPPUCCINO, 0.30),
        (ProductType.LATTE, 0.20),
        (ProductType.AMERICANO, 0.15),
        (ProductType.HOT_WATER, 0.05),
        (ProductType.RISTRETTO, 0.05)
    ])

FUNCTION add_noise(mean, std_dev):
    """Add Gaussian noise to sensor reading"""
    RETURN random_normal(mean, std_dev)

FUNCTION is_business_time(timestamp):
    """Check if timestamp falls within business hours"""
    hour = get_hour(timestamp)
    day = get_day_of_week(timestamp)
    RETURN (day BETWEEN MONDAY AND FRIDAY) AND (hour BETWEEN 6 AND 23)


# ============================================================================
# DATA OUTPUT
# ============================================================================

FUNCTION export_synthetic_data(data_points, format):
    """
    Export generated data in various formats
    """
    IF format == "CSV":
        # Flatten nested structures into tabular format
        flattened = flatten_to_tabular(data_points)
        save_to_csv(flattened, "coffee_machine_data.csv")
    
    ELSE IF format == "JSON":
        # Preserve hierarchical structure
        save_to_json(data_points, "coffee_machine_data.json")
    
    ELSE IF format == "PARQUET":
        # Efficient columnar storage for analytics
        save_to_parquet(data_points, "coffee_machine_data.parquet")
    
    ELSE IF format == "TIMESERIES_DB":
        # Insert into time-series database (InfluxDB, TimescaleDB)
        insert_to_timeseries_db(data_points)

# ============================================================================
# END OF PSEUDOCODE
# ============================================================================